<a href="https://colab.research.google.com/github/vad-source/NLPAPP/blob/main/QA/NLPAPP_IR_based_Extractive_QA.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## NLP APPLICATIONS
**Designed by:** RAJA VADHANA PRABHAKAR  
**Organization:** BITS PILANI WILP  
**Purpose:** Academic Training / Proof of Concept  

---
#### Attribution & AI Disclosure
- **Original Design:** The logic, architecture, and modular structure of this notebook were designed by the author.
- **Development Assistance:** Generative AI (e.g., ChatGPT/Claude/Copilot) was used for coding implementation and debugging support.
- **License:** This work is licensed under the [Apache License 2.0](https://apache.org).

## 0. Knowledge Base

In [ ]:
documents = [
    {"id": 1, "text": "Employees are entitled to 12 casual leave days annually."},
    {"id": 2, "text": "Medical insurance covers employees and dependents."},
    {"id": 3, "text": "Probation period lasts for six months from joining."},
    {"id": 4, "text": "Remote work is allowed two days per week."},
    {"id": 5, "text": "Maternity leave is available for 26 weeks."}
]

qa_test = [
    {"question": "How many casual leaves are provided?","ground_truth": "12 casual leave days annually"},
    {"question": "What is the probation duration?","ground_truth": "six months"},
    {"question": "How long is maternity leave?","ground_truth": "26 weeks"}
]

### 1. Query Processor

In [ ]:
from sentence_transformers import SentenceTransformer
class Embedder:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

    def encode_documents(self, docs):
        texts = [d["text"] for d in docs]
        embeddings = self.model.encode(texts)
        return embeddings

    def encode_query(self, query):
        return self.model.encode([query])[0]

In [ ]:
embedderT = Embedder()
doc_embeddingsT = embedderT.encode_documents(documents)
print("Sample Document : ",documents[0]["text"],"\n\tSize of embeddings",doc_embeddingsT.size,"\n\tEmbedding of Sample Document : ",doc_embeddingsT[0][1],doc_embeddingsT[0][2],doc_embeddingsT[0][3])
query_embeddingT = embedderT.encode_query("How long is maternity leave?")
print("Sample Query : ",qa_test[2]["question"],"\n\tSize of embeddings",query_embeddingT.size,"\n\tEmbedding of Query : ",query_embeddingT[1],query_embeddingT[2],query_embeddingT[3])

## 2. Candidate Retriever

In [ ]:
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class SemanticRetriever:
    def __init__(self, documents, embeddings):
        self.documents = documents
        self.embeddings = embeddings

    def retrieve(self, query_embedding, top_k=2):
        similarities = cosine_similarity([query_embedding],self.embeddings)[0]
        ranked_idx = np.argsort(similarities)[::-1]
        results = []
        for idx in ranked_idx[:top_k]:
            results.append({"document": self.documents[idx],"score": similarities[idx]})
        return results

In [ ]:
retrieverT = SemanticRetriever(documents,doc_embeddingsT)
retrieved_docsT = retrieverT.retrieve(query_embeddingT, top_k=2)
print(retrieved_docsT)

## 3. Answer Generator

In [ ]:
#Learners may use this section to code automated modification/rewrite/reordering/filtering of candidate answers before final answers gets curated
class AnswerGenerator:
    def generate_answer(self, retrieved_docs):
        best_doc = retrieved_docs[0]["document"]["text"]
        return best_doc

In [ ]:
generatorT = AnswerGenerator()
answerT = generatorT.generate_answer(retrieved_docsT)
print(answerT)

## 4. Evaluator

In [ ]:
from sentence_transformers import SentenceTransformer
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np

class Evaluator:
    def __init__(self):
        self.model = SentenceTransformer('all-MiniLM-L6-v2')

    def exact_match(self, pred, gt):
        return int(gt.lower() in pred.lower())

    def semantic_similarity(self, pred, gt):
        emb1 = self.model.encode([pred])[0]
        emb2 = self.model.encode([gt])[0]
        sim = cosine_similarity([emb1], [emb2])[0][0]
        return float(sim)

    def recall_at_k(self, retrieved_docs, gt):
        texts = [d["document"]["text"].lower() for d in retrieved_docs]
        for t in texts:
            if gt.lower() in t:
                return 1
        return 0

    def mrr(self, retrieved_docs, gt):
        for rank, doc in enumerate(retrieved_docs):
            text = doc["document"]["text"].lower()
            if gt.lower() in text:
                return 1 / (rank + 1)
        return 0

    def faithfulness(self, answer, retrieved_context):
        emb1 = self.model.encode([answer])[0]
        emb2 = self.model.encode([retrieved_context])[0]
        score = cosine_similarity([emb1], [emb2])[0][0]
        return float(score)

    def ragas_style_score(self,answer_relevance,context_precision,faithfulness):
        return (answer_relevance + context_precision + faithfulness) / 3

In [ ]:
evaluatorT = Evaluator()

print("\nRetrieval Metrics:")
recallT=evaluatorT.recall_at_k(retrieved_docsT,qa_test[2]["ground_truth"])
print("\n\tRECALL",recallT)
MRRT=evaluatorT.mrr(retrieved_docsT,qa_test[2]["ground_truth"])
print("\n\tMRR",MRRT)

print("Generator Metrics:")
EMT = evaluatorT.exact_match(answerT,qa_test[2]["ground_truth"])
print("\n\tEXACT MATCH",EMT)
SST = evaluatorT.semantic_similarity(answerT,qa_test[2]["ground_truth"])
print("\n\tSEMANTIC SIMILARITY",SST)
FT = evaluatorT.faithfulness(answerT,retrieved_docsT[0]["document"]["text"])
print("\n\tFAITHFULNESS",FT)


##

## Pipeline

In [ ]:
embedder = Embedder()
doc_embeddings = embedder.encode_documents(documents)
retriever = SemanticRetriever( documents,doc_embeddings)
generator = AnswerGenerator()
evaluator = Evaluator()

all_recall = []
all_mrr = []
all_em = []
all_semantic = []
all_ragas = []

for sample in qa_test:
    question = sample["question"]
    ground_truth = sample["ground_truth"]

    query_embedding = embedder.encode_query(question)
    retrieved_docs = retriever.retrieve(query_embedding, top_k=2)
    answer = generator.generate_answer( retrieved_docs)

    recall = evaluator.recall_at_k(retrieved_docs,ground_truth)
    mrr = evaluator.mrr(retrieved_docs, ground_truth)
    em = evaluator.exact_match(answer,ground_truth)
    semantic = evaluator.semantic_similarity(answer, ground_truth)
    context_precision = recall
    faithfulness = evaluator.faithfulness(answer,retrieved_docs[0]["document"]["text"])
    ragas = evaluator.ragas_style_score(semantic,context_precision,faithfulness)


    all_recall.append(recall)
    all_mrr.append(mrr)
    all_em.append(em)
    all_semantic.append(semantic)
    all_ragas.append(ragas)

    print("=" * 50)
    print("QUESTION:", question)
    print("ANSWER:", answer)
    print("GROUND TRUTH:", ground_truth)
    print("\nRetrieved Docs:")
    for d in retrieved_docs:
        print(d)
    print("\nMetrics")
    print("Recall@K:", recall)
    print("MRR:", round(mrr, 3))
    print("Exact Match:", em)
    print("Semantic Similarity:", round(semantic, 3))
    print("Faithfulness:", round(faithfulness, 3))
    print("RAGAS-style:", round(ragas, 3))

print("\n" + "=" * 50)
print("FINAL AGGREGATED METRICS")
print("=" * 50)

print("Avg Recall@K:", round(sum(all_recall)/len(all_recall), 3))
print("Avg MRR:", round(sum(all_mrr)/len(all_mrr), 3))
print("Avg Exact Match:", round(sum(all_em)/len(all_em), 3))
print("Avg Semantic Similarity:",round(sum(all_semantic)/len(all_semantic), 3))
print("Avg RAGAS-style:", round(sum(all_ragas)/len(all_ragas), 3))

In [ ]:
#If you wish to modularize and make each of above function resuable in your local repository, create seperate file . Then you may import them in codes eg., below:
#from data import documents, qa_test
#from embedder import Embedder
#from retriever import SemanticRetriever
#from answer_generator import AnswerGenerator
#from evaluator import Evaluator